# Install & Import

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgbm
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import HyperbandPruner
import optuna
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate,
    plot_slice,
    plot_contour,
)
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    root_mean_squared_error,
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    make_scorer,
    mean_squared_log_error,
)
import scipy.stats as stats

# Settings/Constant variables

In [ ]:
RND_STATE = 37
TRAIN_TEST_SPLIT = 0.05
TRAIN_VALID_SPLIT = 0.05
EARLY_STOPPING_ROUNDS = 50
HYPERPARAM_TUNING_NFOLDS = 10
TERMINATION_OPTUNA = 1.5 * 60 * 60
DATASET_FILENAME = "reg_default_modeling.parquet"

In [ ]:
BEST_PARAMS = {
    "seed": 37,
    "device_type": "gpu",
    "metric": "rmse",
    "first_metric_only": True,
    "objective": "tweedie",
    "n_estimators": 2882,
    "learning_rate": 0.21152141556671036,
    "num_leaves": 156,
    "min_data_in_leaf": 88,
    "max_depth": 94,
    "feature_fraction": 0.7766934871535861,
    "bagging_fraction": 0.7025657080838581,
    "bagging_freq": 21,
    "max_bin": 204,
    "lambda_l2": 7.472679881834768,
    "lambda_l1": 0.5442780255722681,
    "min_sum_hessian_in_leaf": 0.2579903075341818,
    "min_gain_to_split": 0.9817984541790701,
    "feature_fraction_bynode": 0.8078741661167661,
    "path_smooth": 35.136998080140174,
    "extra_trees": True,
    "tweedie_variance_power": 1.1113326530370182,
}

In [ ]:
SCORERS = {
    "rmse": lambda y_true, y_pred, sample_weight=None: np.sqrt(
        mean_squared_error(y_true, y_pred, sample_weight=sample_weight)
    ),
    "mae": lambda y_true, y_pred, sample_weight=None: mean_absolute_error(
        y_true, y_pred, sample_weight=sample_weight
    ),
    "r2": lambda y_true, y_pred, sample_weight=None: r2_score(
        y_true, y_pred, sample_weight=sample_weight
    ),
}

# Read data

In [ ]:
# Read the dataset
modeling = pd.read_parquet(f"data/modeling/{DATASET_FILENAME}")

In [ ]:
X = modeling.drop(["target", "weight"], axis=1)
y = modeling["target"]
w = modeling["weight"]

## Train/test split

In [ ]:
X_train_full, X_test, y_train_full, y_test, w_train_full, w_test = train_test_split(
    X, y, w, test_size=TRAIN_TEST_SPLIT, random_state=RND_STATE
)
X_train_full.shape, X_test.shape, y_train_full.shape, y_test.shape, w_train_full.shape, w_test.shape

## Train/validation split

In [ ]:
X_train, X_val, y_train, y_val, w_train, w_val = train_test_split(
    X_train_full,
    y_train_full,
    w_train_full,
    test_size=TRAIN_VALID_SPLIT,
    random_state=RND_STATE,
)

# Default model

In [ ]:
MODEL_STATIC_PARAMS = {
    "seed": RND_STATE,
    "device_type": "gpu",
    "metric": "rmse",
    "first_metric_only": True,
    "objective": "regression",
}

## Train

In [ ]:
default_lgbm_model = lgbm.LGBMRegressor(**MODEL_STATIC_PARAMS)

default_lgbm_model.fit(
    X_train,
    y_train,
    sample_weight=w_train,
    eval_set=[(X_val, y_val)],
    eval_sample_weight=[w_val],
    callbacks=[
        lgbm.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=True)
    ],
)
joblib.dump(default_lgbm_model, "models/default_reg_model.joblib")

In [ ]:
y_pred = default_lgbm_model.predict(X_test)

# Calculate residuals between actual and predicted values
# residuals = y_test - y_pred
residuals = np.sqrt(w_test) * (y_test - y_pred)

## Examine results

In [ ]:
for name, scorer in SCORERS.items():
    print(f"{name} weighted   = {scorer(y_test, y_pred, sample_weight=w_test):.4f}")

In [ ]:
# Q-Q plot
plt.figure(figsize=(8, 6))
stats.probplot(residuals, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals vs Normal dist")
plt.xlabel("Theoretical Quantiles")
plt.ylabel("Sample Quantiles")
plt.grid(True)
plt.show()

# Actual vs Predicted
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([0, 3], [0, 3], linestyle="--")
plt.xlim(0, 3)
plt.ylim(0, 3)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Scatter Plot: Actual vs Predicted")
plt.grid(True)
plt.show()

# Residuals vs Predicted
plt.figure(figsize=(8, 6))
plt.scatter(y_pred, residuals, alpha=0.7)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Values")
plt.ylabel("Residuals (Actual - Predicted)")
plt.title("Residuals vs Predicted Values")
plt.grid(True)
plt.show()

# Histogram of Residuals
plt.figure(figsize=(8, 6))
plt.hist(residuals, bins=30, edgecolor="k", alpha=0.7)
plt.title("Histogram of Residuals")
plt.xlabel("Residuals")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()

# Feature importance
plt.figure(figsize=(10, 8))
lgbm.plot_importance(default_lgbm_model, importance_type="gain", max_num_features=30)
plt.title("Feature Importance (LightGBM)")
plt.show()

# Optuna hiperparameter optimization

In [ ]:
opt_cv = KFold(n_splits=HYPERPARAM_TUNING_NFOLDS, shuffle=True, random_state=RND_STATE)

## Sampler and pruner definition

In [ ]:
sampler = TPESampler(
    seed=RND_STATE,
    multivariate=True,
    group=True,
    n_startup_trials=50,
    constant_liar=True,
)

pruner = HyperbandPruner(
    min_resource=3,
    max_resource=HYPERPARAM_TUNING_NFOLDS,
    bootstrap_count=25,
    reduction_factor=2,
)

## Define objective

In [ ]:
def objective(trial):
    obj = trial.suggest_categorical(
        "objective",
        [
            "regression",
            "regression_l1",
            "huber",
            "fair",
            "poisson",
            "quantile",
            "mape",
            "gamma",
            "tweedie",
        ],
    )

    params = {
        **MODEL_STATIC_PARAMS,
        "objective": obj,
        # ---- core capacity & learning dynamics
        "n_estimators": trial.suggest_int("n_estimators", 100, 3_000),
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.5, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 2, 256),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 2, 512),
        "max_depth": trial.suggest_int("max_depth", 2, 96),
        # ---- overfitting control via sampling
        "feature_fraction": trial.suggest_float("feature_fraction", 0.1, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.1, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 0, 25),
        # ---- histogram precision
        "max_bin": trial.suggest_int("max_bin", 32, 384),
        # ---- regularization
        "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 100.0),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 100.0),
        "min_sum_hessian_in_leaf": trial.suggest_float(
            "min_sum_hessian_in_leaf", 1e-5, 10.0, log=True
        ),
        "min_gain_to_split": trial.suggest_float("min_gain_to_split", 0.0, 10.0),
        # ---- extra generalization boosters
        "feature_fraction_bynode": trial.suggest_float(
            "feature_fraction_bynode", 0.1, 1.0
        ),
        "path_smooth": trial.suggest_float("path_smooth", 0.0, 100.0),
        "extra_trees": trial.suggest_categorical("extra_trees", [True, False]),
        "verbosity": -1,
    }

    # ---- objective params, ale tylko tam gdzie działają
    if obj == "regression":
        params["boost_from_average"] = trial.suggest_categorical(
            "boost_from_average", [True, False]
        )
        params["reg_sqrt"] = trial.suggest_categorical("reg_sqrt", [True, False])

    # ---- objective-specific params
    if obj == "huber":
        params["alpha"] = trial.suggest_float("alpha", 1e-3, 10.0)  # alpha > 0

    elif obj == "quantile":
        params["alpha"] = trial.suggest_float("alpha", 0.01, 0.99)  # quantile level

    elif obj == "fair":
        params["fair_c"] = trial.suggest_float("fair_c", 1e-3, 100.0, log=True)

    elif obj == "poisson":
        params["poisson_max_delta_step"] = trial.suggest_float(
            "poisson_max_delta_step", 1e-3, 10.0, log=True
        )

    elif obj == "tweedie":
        params["tweedie_variance_power"] = trial.suggest_float(
            "tweedie_variance_power", 1.0, 1.999
        )

    rmse_values, step = [], 1
    for tr_idx, va_idx in opt_cv.split(X_train_full, y_train_full):
        X_fold_tr, X_fold_va = X_train_full.iloc[tr_idx], X_train_full.iloc[va_idx]
        y_fold_tr, y_fold_va = y_train_full.iloc[tr_idx], y_train_full.iloc[va_idx]
        w_fold_tr, w_fold_va = w_train_full.iloc[tr_idx], w_train_full.iloc[va_idx]

        try:
            model = lgbm.LGBMRegressor(**params)
            model.fit(
                X_fold_tr,
                y_fold_tr,
                sample_weight=w_fold_tr,
                eval_set=[(X_fold_va, y_fold_va)],
                eval_sample_weight=[w_fold_va],
                callbacks=[
                    lgbm.early_stopping(
                        stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False
                    )
                ],
            )
            preds = model.predict(X_fold_va)
            rmse = root_mean_squared_error(y_fold_va, preds, sample_weight=w_fold_va)

        except (lgbm.basic.LightGBMError, ValueError):
            raise optuna.TrialPruned()

        rmse_values.append(float(rmse))

        mean_so_far = float(np.mean(rmse_values))
        std_so_far = float(np.std(rmse_values)) if len(rmse_values) > 1 else 0.0
        score_so_far = mean_so_far + std_so_far

        trial.report(score_so_far, step=step)
        if trial.should_prune():
            raise optuna.TrialPruned()
        step += 1

    mean_rmse = float(np.mean(rmse_values))
    std_rmse = float(np.std(rmse_values))
    return mean_rmse + std_rmse

## Study

In [ ]:
study = optuna.create_study(direction="minimize", sampler=sampler, pruner=pruner)

### Inject prev params

In [ ]:
default_params = {
    **MODEL_STATIC_PARAMS,
    "n_estimators": 100,
    "learning_rate": 0.1,
    "num_leaves": 31,
    "max_depth": -1,
    "min_data_in_leaf": 20,
    "feature_fraction": 1.0,
    "bagging_fraction": 1.0,
    "bagging_freq": 0,
    "lambda_l1": 0.0,
    "lambda_l2": 0.0,
    "min_gain_to_split": 0.0,
    "min_sum_hessian_in_leaf": 1e-3,
    "max_bin": 255,
    "feature_fraction_bynode": 1.0,
    "path_smooth": 0,
    "extra_trees": False,
}

In [ ]:
search_space = {
    "objective",
    "n_estimators",
    "learning_rate",
    "num_leaves",
    "max_depth",
    "min_data_in_leaf",
    "feature_fraction",
    "bagging_fraction",
    "bagging_freq",
    "lambda_l1",
    "lambda_l2",
    "min_gain_to_split",
    "tweedie_variance_power",
    "alpha",
    "fair_c",
    "min_sum_hessian_in_leaf",
    "max_bin",
    "feature_fraction_bynode",
    "path_smooth",
    "extra_trees",
}

injected_params = {k: v for k, v in BEST_PARAMS.items() if k in search_space}

# Add conditional keys only when sensowne dla celu
if injected_params.get("objective") != "tweedie":
    injected_params.pop("tweedie_variance_power", None)
if injected_params.get("objective") != "huber":
    injected_params.pop("alpha", None)
if injected_params.get("objective") != "fair":
    injected_params.pop("fair_c", None)

if injected_params:
    study.enqueue_trial(injected_params)

study.enqueue_trial(default_params)

### Run study

In [ ]:
study.optimize(objective, timeout=TERMINATION_OPTUNA, show_progress_bar=True)

In [ ]:
BEST_PARAMS = study.best_trial.params
BEST_PARAMS = {**MODEL_STATIC_PARAMS, **study.best_trial.params}
print("Best params:", BEST_PARAMS)

### Visualize study

In [ ]:
# 1) Historia optymalizacji (wartość celu vs numer triala)
fig_hist = plot_optimization_history(study)
fig_hist.show()

# # 2) Wartości pośrednie (jeśli w objective używałeś trial.report)
# fig_inter = plot_intermediate_values(study)
# fig_inter.show()

# 3) Ważność hiperparametrów (wymaga scikit-learn)
fig_imp = plot_param_importances(study)
fig_imp.show()

# # 4) Równoległe współrzędne – zależności między parametrami a wynikiem
# fig_pc = plot_parallel_coordinate(study)
# fig_pc.show()

# 5) Slice plot – wpływ pojedynczych parametrów na wynik
fig_slice = plot_slice(study)
fig_slice.show()

# # 6) Contour – interakcje param–param
# fig_contour = plot_contour(study)
# fig_contour.show()

# # 7) EDF – rozkład wartości funkcji celu
# fig_edf = plot_edf(study)
# fig_edf.show()

## Train

In [ ]:
opt_lgbm_model = lgbm.LGBMRegressor(**BEST_PARAMS)
opt_lgbm_model.fit(
    X_train,
    y_train,
    sample_weight=w_train,
    eval_set=[(X_val, y_val)],
    eval_sample_weight=[w_val],
    callbacks=[
        lgbm.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=True)
    ],
)

os.makedirs("models", exist_ok=True)
joblib.dump(opt_lgbm_model, "models/opt_reg_model.joblib")

## Examine results

In [ ]:
y_pred = opt_lgbm_model.predict(X_test)

residuals = np.sqrt(w_test) * (y_test - y_pred)

In [ ]:
for name, scorer in SCORERS.items():
    print(f"{name} weighted   = {scorer(y_test, y_pred, sample_weight=w_test):.4f}")

In [ ]:
rmse = root_mean_squared_error(y_test, y_pred, sample_weight=w_test)
mae = mean_absolute_error(y_test, y_pred, sample_weight=w_test)
r2 = r2_score(y_test, y_pred, sample_weight=w_test)
DF_SCORES = pd.DataFrame({"Type": ["opt"], "RMSE": [rmse], "MAE": [mae], "R2": [r2]})
print(DF_SCORES)

In [ ]:
# Q-Q plot
plt.figure(figsize=(8, 6))
stats.probplot(residuals, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals vs Normal dist")
plt.xlabel("Theoretical Quantiles")
plt.ylabel("Sample Quantiles")
plt.grid(True)
plt.show()

# Actual vs Predicted
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([0, 3], [0, 3], linestyle="--")
plt.xlim(0, 3)
plt.ylim(0, 3)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Scatter Plot: Actual vs Predicted")
plt.grid(True)
plt.show()

# Residuals vs Predicted
plt.figure(figsize=(8, 6))
plt.scatter(y_pred, residuals, alpha=0.7)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Values")
plt.ylabel("Residuals (Actual - Predicted)")
plt.title("Residuals vs Predicted Values")
plt.grid(True)
plt.show()

# Histogram of Residuals
plt.figure(figsize=(8, 6))
plt.hist(residuals, bins=30, edgecolor="k", alpha=0.7)
plt.title("Histogram of Residuals")
plt.xlabel("Residuals")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()

# Feature importance
plt.figure(figsize=(10, 8))
lgbm.plot_importance(opt_lgbm_model, importance_type="gain", max_num_features=30)
plt.title("Feature Importance (LightGBM)")
plt.show()

## Quick comparison default vs optimized

In [ ]:
def cv_rmse(params):
    rmses = []
    for tr_idx, va_idx in opt_cv.split(X_train_full, y_train_full):
        X_tr, X_va = X_train_full.iloc[tr_idx], X_train_full.iloc[va_idx]
        y_tr, y_va = y_train_full.iloc[tr_idx], y_train_full.iloc[va_idx]
        w_tr, w_va = w_train_full.iloc[tr_idx], w_train_full.iloc[va_idx]

        m = lgbm.LGBMRegressor(**params)
        m.fit(
            X_tr,
            y_tr,
            sample_weight=w_tr,
            eval_set=[(X_va, y_va)],
            eval_sample_weight=[w_va],
            callbacks=[lgbm.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)],
        )
        p = m.predict(X_va)
        rmses.append(root_mean_squared_error(y_va, p, sample_weight=w_va))
    return float(np.mean(rmses)), float(np.std(rmses))


default_params_for_cv = {
    **MODEL_STATIC_PARAMS
}  # czyli czysty default LGBM + Twoje metric/seed/gpu
best_params_for_cv = BEST_PARAMS

print("default CV rmse (mean,std):", cv_rmse(default_params_for_cv))
print("best    CV rmse (mean,std):", cv_rmse(best_params_for_cv))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
import numpy as np


def eval_once(params, seed):
    X_tr, X_te, y_tr, y_te, w_tr, w_te = train_test_split(
        X, y, w, test_size=0.05, random_state=seed
    )
    X_tr2, X_va, y_tr2, y_va, w_tr2, w_va = train_test_split(
        X_tr, y_tr, w_tr, test_size=0.10, random_state=seed
    )

    m = lgbm.LGBMRegressor(**params)
    m.fit(
        X_tr2,
        y_tr2,
        sample_weight=w_tr2,
        eval_set=[(X_va, y_va)],
        eval_sample_weight=[w_va],
        callbacks=[lgbm.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)],
    )
    p = m.predict(X_te)
    return root_mean_squared_error(y_te, p, sample_weight=w_te)


seeds = range(10, 40)  # 30 losowań
rmse_def = np.array([eval_once(MODEL_STATIC_PARAMS, s) for s in seeds])
rmse_opt = np.array([eval_once(BEST_PARAMS, s) for s in seeds])

print("default mean/std:", rmse_def.mean(), rmse_def.std())
print("opt     mean/std:", rmse_opt.mean(), rmse_opt.std())
print("opt wins %:", (rmse_opt < rmse_def).mean())

# Final model

In [ ]:
(
    X_train_final,
    X_test_final,
    y_train_final,
    y_test_final,
    w_train_final,
    w_test_final,
) = train_test_split(X, y, w, test_size=0.01, random_state=RND_STATE)

In [ ]:
X_tr_f, X_val_f, y_tr_f, y_val_f, w_tr_f, w_val_f = train_test_split(
    X_train_final,
    y_train_final,
    w_train_final,
    test_size=TRAIN_VALID_SPLIT,
    random_state=RND_STATE,
)

## Train

In [ ]:
if BEST_PARAMS is not None:
    final_lgbm_model = lgbm.LGBMRegressor(**BEST_PARAMS, verbosity=-1)
else:
    final_lgbm_model = lgbm.LGBMRegressor(**MODEL_STATIC_PARAMS, verbosity=-1)

final_lgbm_model.fit(
    X_tr_f,
    y_tr_f,
    sample_weight=w_tr_f,
    eval_set=[(X_val_f, y_val_f)],
    eval_sample_weight=[w_val_f],
    callbacks=[
        lgbm.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=True)
    ],
)
joblib.dump(final_lgbm_model, "models/final_reg_model.joblib")

In [ ]:
y_pred = final_lgbm_model.predict(X_test_final)

# Calculate residuals between actual and predicted values
residuals = np.sqrt(w_test_final) * (y_test_final - y_pred)

## Examine results

In [ ]:
for name, scorer in SCORERS.items():
    print(
        f"{name} weighted   = {scorer(y_test_final, y_pred, sample_weight=w_test_final):.4f}"
    )

In [ ]:
# sorted(y_pred, reverse=True)
np.quantile(y_pred, [0.25, 0.5, 0.75, 0.95, 0.99, 1.0])

In [ ]:
np.mean(y_pred)

In [ ]:
# Q-Q plot
plt.figure(figsize=(8, 6))
stats.probplot(residuals, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals vs Normal dist")
plt.xlabel("Theoretical Quantiles")
plt.ylabel("Sample Quantiles")
plt.grid(True)
plt.show()

# Actual vs Predicted
plt.figure(figsize=(8, 6))
plt.scatter(y_test_final, y_pred, alpha=0.7)
plt.plot([0, 3], [0, 3], linestyle="--")
plt.xlim(0, 3)
plt.ylim(0, 3)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Scatter Plot: Actual vs Predicted")
plt.grid(True)
plt.show()

# Residuals vs Predicted
plt.figure(figsize=(8, 6))
plt.scatter(y_pred, residuals, alpha=0.7)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Values")
plt.ylabel("Residuals (Actual - Predicted)")
plt.title("Residuals vs Predicted Values")
plt.grid(True)
plt.show()

# Histogram of Residuals
plt.figure(figsize=(8, 6))
plt.hist(residuals, bins=30, edgecolor="k", alpha=0.7)
plt.title("Histogram of Residuals")
plt.xlabel("Residuals")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()

# Feature importance
plt.figure(figsize=(10, 8))
lgbm.plot_importance(final_lgbm_model, importance_type="gain", max_num_features=30)
plt.title("Feature Importance (LightGBM)")
plt.show()